# Milestone 2 — Enhancer CPU Benchmark

Measures per-frame inference time for `Enhancer.upscale_frame` and `Enhancer.upscale_roi`
across multiple resolutions and ROI sizes.

**Author:** Victor Teixeira  
**Goal:** Determine whether `--enhance` is viable for real-time sources and estimate
the overhead added per frame in the pipeline.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
for p in [str(repo_root), str(repo_root / 'src'), str(repo_root / 'scripts')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Repo root:', repo_root)

In [ ]:
from benchmark_enhancer import (
    benchmark_upscale_frame,
    benchmark_upscale_roi,
    RESOLUTIONS,
    ROI_SIZES,
)
from src.enhancement.enhancer import Enhancer
import pandas as pd
import matplotlib.pyplot as plt

## Setup

The `Enhancer` will use Real-ESRGAN if `models/RealESRGAN_x4plus.pth` is present,
otherwise it falls back to bicubic interpolation. Both are benchmarked the same way.

In [ ]:
N_FRAMES = 20   # timed calls per configuration (increase for more stable estimates)
SCALE    = 4

enhancer = Enhancer(scale=SCALE)
print(f'Backend : {enhancer.backend}')
print(f'Scale   : {enhancer.scale}x')
print(f'Model   : {enhancer.model_path}')

## 1 — upscale_frame timing

In [ ]:
frame_results = benchmark_upscale_frame(enhancer, N_FRAMES)
df_frame = pd.DataFrame(frame_results)
df_frame[['resolution', 'frame_size', 'backend', 'mean_ms', 'min_ms', 'max_ms', 'std_ms']]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
labels = df_frame['resolution'].tolist()
means  = df_frame['mean_ms'].tolist()
stds   = df_frame['std_ms'].tolist()

bars = ax.bar(labels, means, yerr=stds, capsize=5, color='steelblue', alpha=0.8)
ax.set_xlabel('Resolution')
ax.set_ylabel('Mean inference time (ms)')
ax.set_title(f'upscale_frame — CPU time by resolution  (backend: {enhancer.backend}, x{SCALE})')

# Annotate each bar with the mean value
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{mean:.0f} ms', ha='center', va='bottom', fontsize=9)

# Draw a 33 ms line = max budget for 30 fps real-time
ax.axhline(33, color='red', linestyle='--', linewidth=1, label='33 ms (30 fps budget)')
ax.legend()
plt.tight_layout()
plt.savefig(repo_root / 'outputs' / 'enhancer_frame_timing.png', dpi=120)
plt.show()
print('Saved: outputs/enhancer_frame_timing.png')

## 2 — upscale_roi timing

In [ ]:
roi_results = benchmark_upscale_roi(enhancer, N_FRAMES)
df_roi = pd.DataFrame(roi_results)
df_roi[['resolution', 'roi_size', 'backend', 'mean_ms', 'min_ms', 'max_ms']]

In [ ]:
fig, axes = plt.subplots(1, len(RESOLUTIONS), figsize=(14, 4), sharey=True)

for ax, (res_label, w, h) in zip(axes, RESOLUTIONS):
    subset = df_roi[df_roi['resolution'] == res_label]
    roi_labels = [r.strip() for r in subset['roi_size'].tolist()]
    means = subset['mean_ms'].tolist()
    stds  = subset['std_ms'].tolist()

    bars = ax.bar(roi_labels, means, yerr=stds, capsize=5, color='darkorange', alpha=0.8)
    ax.set_title(f'{res_label} ({w}x{h})')
    ax.set_xlabel('ROI size')
    if ax is axes[0]:
        ax.set_ylabel('Mean inference time (ms)')

    ax.axhline(33, color='red', linestyle='--', linewidth=1)

    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{mean:.0f}', ha='center', va='bottom', fontsize=8)

fig.suptitle(f'upscale_roi — CPU time by resolution & ROI size  (backend: {enhancer.backend}, x{SCALE})', y=1.02)
plt.tight_layout()
plt.savefig(repo_root / 'outputs' / 'enhancer_roi_timing.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: outputs/enhancer_roi_timing.png')

## 3 — Real-time feasibility

In [ ]:
FPS_BUDGET_MS = 1000 / 30   # 33.3 ms per frame at 30 fps

print(f'30 fps frame budget: {FPS_BUDGET_MS:.1f} ms\n')
print(f'{"Resolution":<10} {"Mean ms":>9} {"Real-time viable?":>20}')
print('-' * 42)
for _, row in df_frame.iterrows():
    viable = 'YES' if row['mean_ms'] < FPS_BUDGET_MS else 'NO'
    print(f"{row['resolution']:<10} {row['mean_ms']:>9.1f} {viable:>20}")

print()
print('Note: upscale_roi cost scales with ROI size, not full frame.')
print('For typical foreground coverage (~10%), upscale_roi is cheaper than upscale_frame.')

## 4 — Summary

| | |
|---|---|
| **Backend** | bicubic (no weights) or Real-ESRGAN (with weights) |
| **Recommendation** | Use `--enhance` for **offline processing only** unless CPU timing shows < 33 ms at your target resolution |
| **Tip** | Pass `--enhance-scale 2` to use a lighter upscale pass for faster throughput |